In [27]:
## Install Coqui TTS
! pip install -U pip
! pip install TTS

  Using cached gruut-2.2.3-py3-none-any.whl
  Attempting uninstall: gruut
    Found existing installation: gruut 2.4.0
    Uninstalling gruut-2.4.0:
      Successfully uninstalled gruut-2.4.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
coqui-tts 0.25.3 requires gruut[de,es,fr]>=2.4.0, but you have gruut 2.2.3 which is incompatible.


In [6]:
import sys
print(sys.executable)  # Should show /xtts-vc path
from TTS.api import TTS
print(" XTTS kernel perfect!")

/usr/bin/python3
 XTTS kernel perfect!


In [7]:
import soundfile as sf
import numpy as np
import os
from pathlib import Path
from scipy.signal import resample


input_dir = Path.cwd() / "input_wavs"
input_dir.mkdir(exist_ok=True)

In [8]:
original_files = [
    'recording2.wav','recording1.wav','recording4.wav'
]

# XTTS-optimized output files (in input_wavs/ subdirectory)
xtts_files = [input_dir / f"{Path(f).stem}_xtts.wav" for f in original_files]

print("CONVERTING TO XTTS FORMAT (Mono 22050Hz)...\n")

for orig_name, xtts_path in zip(original_files, xtts_files):
    orig_path = Path(orig_name)  # Original in cwd

    if not orig_path.exists():
        print(f"✗ {orig_name:12} not found - skipping")
        continue

    # Load original
    print(f"Processing {orig_name}...")
    audio, sr = sf.read(orig_path)

    # Convert to mono (average channels if stereo)
    if len(audio.shape) > 1:
        audio = np.mean(audio, axis=1)
    else:
        audio = audio.flatten()

    # Resample to 22050Hz
    target_sr = 22050
    if sr != target_sr:
        num_samples = int(len(audio) * target_sr / sr)
        audio = resample(audio, num_samples)

    # Normalize to prevent clipping
    if np.max(np.abs(audio)) > 0:
        audio = audio / np.max(np.abs(audio)) * 0.95

    # Save XTTS-ready file to input_wavs/ subdirectory
    sf.write(xtts_path, audio, target_sr)

    duration = len(audio) / target_sr
    print(f"✓ {orig_name:12} → {xtts_path.name:20} "
          f"({duration:.1f}s, {target_sr}Hz, mono)")

# Update input_files list for XTTS usage (full paths)
input_files = [p for p in xtts_files if p.exists()]
print(f"All files ready in XTTS format: {len(input_files)} files")
print(f"Input directory: {input_dir.absolute()}")


print("XTTS-READY FILE ANALYSIS:")
print(f"Working from: {Path.cwd()}")
print(f"Input dir:    {input_dir.absolute()}\n")

for f in input_files:
    audio, sr = sf.read(f)
    duration = len(audio) / sr
    print(f"  ✓ {f.name:20} | {sr:6}Hz | Mono | {duration:4.1f}s | {len(audio):,} samples")

total_duration = sum(sf.info(f).duration for f in input_files)
print(f"\nTotal duration: {total_duration:.1f}s across {len(input_files)} XTTS-ready files")

CONVERTING TO XTTS FORMAT (Mono 22050Hz)...

Processing recording2.wav...
✓ recording2.wav → recording2_xtts.wav  (7.4s, 22050Hz, mono)
Processing recording1.wav...
✓ recording1.wav → recording1_xtts.wav  (5.5s, 22050Hz, mono)
Processing recording4.wav...
✓ recording4.wav → recording4_xtts.wav  (6.2s, 22050Hz, mono)
All files ready in XTTS format: 3 files
Input directory: /content/input_wavs
XTTS-READY FILE ANALYSIS:
Working from: /content
Input dir:    /content/input_wavs

  ✓ recording2_xtts.wav  |  22050Hz | Mono |  7.4s | 163,699 samples
  ✓ recording1_xtts.wav  |  22050Hz | Mono |  5.5s | 122,304 samples
  ✓ recording4_xtts.wav  |  22050Hz | Mono |  6.2s | 135,945 samples

Total duration: 19.1s across 3 XTTS-ready files


In [9]:
import os
os.makedirs('input_wavs', exist_ok=True)
os.makedirs('output_cloned', exist_ok=True)

print("Setup complete. Add your WAV files to input_audio/")

Setup complete. Add your WAV files to input_audio/


In [10]:
input_files = [
    'input_wavs/recording1_xtts.wav',
    'input_wavs/recording2_xtts.wav',
    'input_wavs/recording4_xtts.wav',
]

# Verify files exist and get durations
import soundfile as sf
import numpy as np

file_info = {}
total_duration = 0
for f in input_files:
    if os.path.exists(f):
        audio, sr = sf.read(f)
        duration = len(audio) / sr
        file_info[f] = {'duration': duration, 'sr': sr, 'samples': len(audio)}
        total_duration += duration
        print(f"✓ {os.path.basename(f)}: {duration:.1f}s, {sr}Hz")
    else:
        print(f"✗ {f} not found - please add to input_wavs/")

print(f"\nTotal audio: {total_duration:.1f}s across {len(file_info)} files")

✓ recording1_xtts.wav: 5.5s, 22050Hz
✓ recording2_xtts.wav: 7.4s, 22050Hz
✓ recording4_xtts.wav: 6.2s, 22050Hz

Total audio: 19.1s across 3 files


In [11]:
!pip install "transformers<=4.44.2"

In [12]:
import transformers
print(f"Transformers version: {transformers.__version__}")  # Must be <=4.44.2

from TTS.api import TTS
print(" XTTSv2 imports successfully!")

Transformers version: 4.44.2
 XTTSv2 imports successfully!


In [13]:
!pip install "coqui-tts<=0.25.3" "coqui-tts-trainer<=0.2.2"

In [14]:
import subprocess
import sys

print("=== Package Versions ===")
result = subprocess.run([sys.executable, '-m', 'pip', 'list'],
                       capture_output=True, text=True)
packages = ['TTS', 'coqui-tts', 'transformers', 'gruut', 'torch']
for line in result.stdout.split('\n'):
    for pkg in packages:
        if pkg.lower() in line.lower():
            print(line.strip())

# Test import
try:
    from TTS.api import TTS
    print("\nTTS imported successfully!")
except ImportError as e:
    print(f"\nImport error: {e}")

=== Package Versions ===
coqui-tts                             0.25.3
coqui-tts                             0.25.3
coqui-tts-trainer                     0.2.2
coqui-tts-trainer                     0.2.2
gruut                                 2.4.0
gruut-ipa                             0.13.0
gruut_lang_de                         2.0.1
gruut_lang_en                         2.0.1
gruut_lang_es                         2.0.1
gruut_lang_fr                         2.0.2
sentence-transformers                 4.1.0
torch                                 2.6.0+cu124
torchao                               0.10.0
torchaudio                            2.6.0+cu124
torchdata                             0.11.0
torchsummary                          1.5.1
torchtune                             0.6.1
torchvision                           0.21.0+cu124
transformers                          4.44.2
TTS                                   0.22.0

TTS imported successfully!


In [15]:
import torch
import torchaudio
from TTS.api import TTS
import IPython.display as ipd
import os

# Check device
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

tts = TTS("tts_models/multilingual/multi-dataset/xtts_v2").to(device)
print("XTTSv2 model loaded!")

# Load multilingual XTTS-v2 (~1.8GB)
tts = TTS("/mnt/c/Users/pow2/0_Notebooks/cscie104/L25/TTS_L26/TTS/TTS/tts/tts_models/multilingual/multi-dataset/xtts_v2").to(device)
print("✓ XTTS-v2 loaded - ready for zero-shot cloning")

Using device: cuda
XTTSv2 model loaded!
✓ XTTS-v2 loaded - ready for zero-shot cloning


In [16]:
print(TTS().list_models())

['tts_models/multilingual/multi-dataset/xtts_v2', 'tts_models/multilingual/multi-dataset/xtts_v1.1', 'tts_models/multilingual/multi-dataset/your_tts', 'tts_models/multilingual/multi-dataset/bark', 'tts_models/bg/cv/vits', 'tts_models/cs/cv/vits', 'tts_models/da/cv/vits', 'tts_models/et/cv/vits', 'tts_models/ga/cv/vits', 'tts_models/en/ek1/tacotron2', 'tts_models/en/ljspeech/tacotron2-DDC', 'tts_models/en/ljspeech/tacotron2-DDC_ph', 'tts_models/en/ljspeech/glow-tts', 'tts_models/en/ljspeech/speedy-speech', 'tts_models/en/ljspeech/tacotron2-DCA', 'tts_models/en/ljspeech/vits', 'tts_models/en/ljspeech/vits--neon', 'tts_models/en/ljspeech/fast_pitch', 'tts_models/en/ljspeech/overflow', 'tts_models/en/ljspeech/neural_hmm', 'tts_models/en/vctk/vits', 'tts_models/en/vctk/fast_pitch', 'tts_models/en/sam/tacotron-DDC', 'tts_models/en/blizzard2013/capacitron-t2-c50', 'tts_models/en/blizzard2013/capacitron-t2-c150_v2', 'tts_models/en/multi-dataset/tortoise-v2', 'tts_models/en/jenny/jenny', 'tts_m

In [17]:
from pathlib import Path

print("VARIABLE STATUS:")
print(f"input_files defined: {'input_files' in globals()}")
print(f"input_files length: {len(input_files) if 'input_files' in globals() else 'N/A'}")

if 'input_files' in globals():
    print("\nFirst 3 files:")
    for f in input_files[:3]:
        print(f"  ✓ {Path(f).name}")
else:
    print("\ninput_files missing! Run Cell 0 (preprocessing) first.")
    print("input_files should contain: input_wavs/recording2.wav, etc.")

VARIABLE STATUS:
input_files defined: True
input_files length: 3

First 3 files:
  ✓ recording1_xtts.wav
  ✓ recording2_xtts.wav
  ✓ recording4_xtts.wav


In [18]:
print("Extracting speaker embedding from 3 reference files...")

# Use ONLY speaker_wav - let XTTS handle embedding internally
clone_test_text = "Voice cloning verification from three audio samples."
reference_wav = input_files[0]  # First XTTS-processed file

print(f"Using reference: {Path(reference_wav).name}")

# CORRECT: No 'speaker=' parameter for initial cloning
tts.tts_to_file(
    text=clone_test_text,
    speaker_wav=reference_wav,  # Extracts embedding automatically
    language="en",
    file_path="output_cloned/clone_verification.wav"
)

print("✓ Speaker embedding extracted!")
print("Generated: output_cloned/clone_verification.wav")

Extracting speaker embedding from 3 reference files...
Using reference: recording1_xtts.wav
✓ Speaker embedding extracted!
Generated: output_cloned/clone_verification.wav


In [19]:
print("Testing cloned voice...")

test_sentences = [
    "This voice was cloned from seven short audio samples using XTTS-v2 zero-shot cloning.",
    "Testing natural prosody, emotion transfer, and long-form generation capability.",
    "With the noise in Auto Encoders, we will have an image and perfect image, and then they will add noise to that image. And then we will push that noise, image with noise, through Auto Encoder. Auto Encoder and Coder of Auto Encoder will create a Latent space, and because Latent space is small, small dimensionally, it will remember only important feature on the image, and it will basically ignore the noise. And then we will expand Latent space image into a real-sized image. And what will come out will be a new image, and we will compare that new image with the original image without noise. So in that way, Auto Encoder will learn to remove noise, because it will be asked to compare noise image or to transform noise image into perfect image."
]

reference_wav = input_files[0]  # ALWAYS use reference for consistency

for i, text in enumerate(test_sentences):
    output_path = f"output_cloned/test_sample_{i+1:02d}.wav"
    tts.tts_to_file(
        text=text,
        speaker_wav=reference_wav,  # Reference audio EVERY generation
        language="en",
        file_path=output_path,
        temperature=0.65,       # Creativity (0.1-1.0)
        length_penalty=1.0,     # Normal length
        repetition_penalty=5.0  # Anti-looping
    )
    print(f"✓ {output_path}")

print("\nAll 3 test samples generated!")

Testing cloned voice...
✓ output_cloned/test_sample_01.wav
✓ output_cloned/test_sample_02.wav
✓ output_cloned/test_sample_03.wav

All 3 test samples generated!


In [20]:
# Play first sentence:
# "This voice was cloned from six short audio samples using XTTS-v2 zero-shot cloning."
ipd.Audio("output_cloned/test_sample_01.wav")

In [21]:
# Play 2nd sentence:
# "This voice was cloned from seven short audio samples using XTTS-v2 zero-shot cloning."
ipd.Audio("output_cloned/test_sample_02.wav")

In [22]:
# Play 3rd sentence:
# "With the noise in Auto Encoders, we will have an image and perfect image, and then they will add noise to that image"
ipd.Audio("output_cloned/test_sample_03.wav")

Output hidden; open in https://colab.research.google.com to view.

In [24]:
from IPython.display import Audio, display
from pathlib import Path

tts = TTS("/mnt/c/Users/pow2/0_Notebooks/cscie104/L25/TTS_L26/TTS/TTS/tts/tts_models/multilingual/multi-dataset/xtts_v2").to(device)
print("✓ XTTS-v2 loaded - ready for zero-shot cloning")

# Use your reference WAV (from input_files)
reference_wav = str(input_files[0])  # Convert Path to string

# Proper French text + syntax + Phonemes
tts.tts_to_file(
    text="Cette voix a été clonée à partir de sept échantillons audio courts en utilisant XTTSv2",
    file_path="output_cloned/spanish_clone.wav",
    speaker_wav=reference_wav,
    language="es"
)

print("Spanish clone generated!")
Audio("output_cloned/spanish_clone.wav")

✓ XTTS-v2 loaded - ready for zero-shot cloning
Spanish clone generated!


In [25]:
tts = TTS('tts_models/multilingual/multi-dataset/your_tts').to(device)
print("your_tts model loaded!")

your_tts model loaded!


In [28]:
print("Extracting speaker embedding from 3 reference files...")

# Use ONLY speaker_wav - let XTTS handle embedding internally
clone_test_text = "Voice cloning verification from three audio samples using a different model."
reference_wav = input_files[0]  # First XTTS-processed file

print(f"Using reference: {Path(reference_wav).name}")

# CORRECT: No 'speaker=' parameter for initial cloning
tts.tts_to_file(
    text=clone_test_text,
    speaker_wav=reference_wav,  # Extracts embedding automatically
    language="en",
    file_path="output_cloned/clone_verification_different_model.wav"
)

print("✓ Speaker embedding extracted!")
print("Generated: output_cloned/clone_verification_different_model.wav")

Extracting speaker embedding from 3 reference files...
Using reference: recording1_xtts.wav
✓ Speaker embedding extracted!
Generated: output_cloned/clone_verification_different_model.wav


In [29]:
Audio("output_cloned/clone_verification_different_model.wav")

In [30]:
from IPython.display import Audio, display
from pathlib import Path

# Use your reference WAV (from input_files)
reference_wav = str(input_files[0])  # Convert Path to string

# Proper portugese text + syntax + Phonemes
tts.tts_to_file(
    text="Cette voix a été clonée à partir de sept échantillons audio courts en utilisant XTTSv2",
    file_path="output_cloned/portugese_clone_model2.wav",
    speaker_wav=reference_wav,
    language="pt-br"
)

print("portugese clone generated!")
Audio("output_cloned/portugese_clone_model2.wav")

portugese clone generated!


I noticed xtt has more noise that your_tts